# Veri Bilimi Eğitimi


## Aykırı Değerler (Outliers)
Amaç:
  1. Veri setindeki aykırı değerlerin tespit edilmesi ve incelenmesi
  2. Aykırı değerleri temizleyerek güncel veri setinin oluşturulması

Adımlar:
  1. Veri setini yükle
  2. Sayısal sütunları belirle
  3. Sayısal sütunların özet istatistiklerini hesapla
  4. IQR ile aykırı değer sınırlarını hesapla
  5. Aykırı değer içeren kayıtları tespit etmek
  6. Aykırı değer sayılarını sütun bazında incelemek
  7. Aykırı değer içeren kayıtları veri setinden temizlemek
  8. Temizlenmiş veri setini kaydetmek

  

In [ ]:
import pandas as pd

In [ ]:
# 1. veri setini yüklemek
df = pd.read_csv("e_ticaret_veri_seti_tekrarlayan_kayitlar_duzenlendi.csv")

In [ ]:
# 2. sayısal sütunların belirlenmesi
sayisal_sutunlar = df.select_dtypes(include = ["int64", "float64"]).columns.tolist()
sayisal_sutunlar

['adet',
 'birim_fiyat',
 'indirim_orani',
 'kargo_ucreti',
 'teslimat_gunu',
 'musteri_puani',
 'kar_marji_orani',
 'toplam_tutar',
 'siparis_yili',
 'siparis_ayi',
 'siparis_gunu']

In [ ]:
# 3. sayısal sütunların özet istatistiklerini inceleyelim
df[sayisal_sutunlar].describe()

,adet,birim_fiyat,indirim_orani,kargo_ucreti,teslimat_gunu,musteri_puani,kar_marji_orani,toplam_tutar,siparis_yili,siparis_ayi,siparis_gunu
count,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000,1398.000000
mean,2.143062,660.712818,0.051717,39.728398,3.355508,3.997854,0.302446,1377.934642,2024.528612,6.394134,15.655222
std,2.031849,1051.014558,0.059681,14.188925,1.771142,1.071193,0.069861,2988.042747,0.499359,3.455187,8.720828
min,1.000000,93.110000,0.000000,0.000000,1.000000,0.000000,0.180000,89.680000,2024.000000,1.000000,1.000000
25%,1.000000,198.855000,0.000000,29.900000,2.000000,4.000000,0.243250,327.190000,2024.000000,3.000000,8.000000
50%,2.000000,287.120000,0.050000,39.900000,3.000000,4.000000,0.305000,670.965000,2025.000000,6.000000,16.000000
75%,3.000000,817.335000,0.100000,49.900000,4.000000,5.000000,0.362000,1393.250000,2025.000000,9.000000,23.000000
max,38.000000,18145.230000,0.200000,59.900000,31.000000,5.000000,0.420000,63870.200000,2025.000000,12.000000,31.000000


In [ ]:
"""birim_fiyat

  IQR = Q3(75%) - Q1(25%)
      = 817 - 198 = 619

  Üst sınır = Q3 + k*IQR
  Alt sınır = Q1 - k*IQR
    k = 1 olsun (1.5 alınır işlem kolaylığı için 1 aldık)

    Üst sınır = 817 + 619 = 1436
    Alt sınır = 198 - 619 = -421
  Outlier = üst sınır ve alt sınır dışında kalan örnekler
"""


'birim_fiyat\n  \n  IQR = Q3(75%) - Q1(25%)\n      = 817 - 198 = 619\n\n  Üst sınır = Q3 + k*IQR\n  Alt sınır = Q1 - k*IQR\n    k = 1 olsun (1.5 alınır işlem kolaylığı için 1 aldık)\n\n    Üst sınır = 817 + 619 = 1436\n    Alt sınır = 198 - 619 = -421\n  Outlier = üst sınır ve alt sınır dışında kalan örnekler\n'

In [ ]:
# 4. IQR ile aykırı değer sınırlarını hesaplama
aykiri_sinirlar = {}

for sutun in sayisal_sutunlar:

  q1 = df[sutun].quantile(0.25)
  q3 = df[sutun].quantile(0.75)

  iqr = q3 - q1
  alt_sinir = q1 - 1.5 * iqr
  ust_sinir = q3 + 1.5 * iqr

  aykiri_sinirlar[sutun] = { #her sayısal sütun için aykırı değerleri yazdırıyoruz
      "q1": q1,
      "q3": q3,
      "iqr": iqr,
      "alt_sinir": alt_sinir,
      "ust_sinir": ust_sinir
  }

In [ ]:
aykiri_sinirlar

{'adet': {'q1': np.float64(1.0),
  'q3': np.float64(3.0),
  'iqr': np.float64(2.0),
  'alt_sinir': np.float64(-2.0),
  'ust_sinir': np.float64(6.0)},
 'birim_fiyat': {'q1': np.float64(198.85500000000002),
  'q3': np.float64(817.335),
  'iqr': np.float64(618.48),
  'alt_sinir': np.float64(-728.865),
  'ust_sinir': np.float64(1745.055)},
 'indirim_orani': {'q1': np.float64(0.0),
  'q3': np.float64(0.1),
  'iqr': np.float64(0.1),
  'alt_sinir': np.float64(-0.15000000000000002),
  'ust_sinir': np.float64(0.25)},
 'kargo_ucreti': {'q1': np.float64(29.9),
  'q3': np.float64(49.9),
  'iqr': np.float64(20.0),
  'alt_sinir': np.float64(-0.10000000000000142),
  'ust_sinir': np.float64(79.9)},
 'teslimat_gunu': {'q1': np.float64(2.0),
  'q3': np.float64(4.0),
  'iqr': np.float64(2.0),
  'alt_sinir': np.float64(-1.0),
  'ust_sinir': np.float64(7.0)},
 'musteri_puani': {'q1': np.float64(4.0),
  'q3': np.float64(5.0),
  'iqr': np.float64(1.0),
  'alt_sinir': np.float64(2.5),
  'ust_sinir': np.float6

In [ ]:
# 5. aykırı değer içeren kayıtları tespit et
aykiri_maskesi = pd.Series(False, index = df.index)

for sutun in sayisal_sutunlar:
  alt_sinir = aykiri_sinirlar[sutun]["alt_sinir"]
  ust_sinir = aykiri_sinirlar[sutun]["ust_sinir"]

  sutun_maskesi = (df[sutun] < alt_sinir) | (df[sutun] > ust_sinir)
  aykiri_maskesi = aykiri_maskesi | sutun_maskesi

In [ ]:
aykiri_kayitlar = df[aykiri_maskesi]
aykiri_kayitlar

,siparis_id,musteri_id,siparis_tarihi,sehir,bolge,kategori,urun_adi,adet,birim_fiyat,indirim_orani,...,musteri_tipi,teslimat_gunu,musteri_puani,iade_durumu,kar_marji_orani,toplam_tutar,siparis_yili,siparis_ayi,siparis_gunu,haftanin_gunu
1,SIP100837,MUS1347,2025-03-17 21:13:00,Gaziantep,Güneydoğu Anadolu,Spor,Dambıl Seti,2,1475.15,0.10,...,VIP,4.0,1.0,Hayır,0.361,2685.17,2025,3,17,Monday
2,SIP100050,MUS1066,2025-06-03 12:42:00,Kayseri,İç Anadolu,Kitap,İş Analitiği Rehberi,1,289.02,0.05,...,Mevcut,6.0,2.0,Hayır,0.348,304.47,2025,6,3,Tuesday
6,SIP101351,MUS1011,2025-02-27 11:35:00,İstanbul,Marmara,Ofis,Mouse Pad,3,169.47,0.05,...,VIP,3.0,2.0,Evet,0.382,512.89,2025,2,27,Thursday
7,SIP101172,MUS1357,2024-07-22 13:26:00,Ankara,İç Anadolu,Ofis,Mekanik Klavye,3,2344.47,0.00,...,Mevcut,3.0,4.0,Hayır,0.312,7093.31,2024,7,22,Monday
8,SIP100779,MUS1354,2025-12-05 11:19:00,İstanbul,Marmara,Kitap,Python Notları,4,219.14,0.00,...,Yeni,18.0,5.0,Evet,0.408,906.46,2025,12,5,Friday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1387,SIP100177,MUS1075,2024-01-29 13:59:00,Trabzon,Karadeniz,Ofis,Mekanik Klavye,3,2169.51,0.00,...,Mevcut,1.0,4.0,Hayır,0.273,6548.43,2024,1,29,Monday
1389,SIP100056,MUS1238,2024-04-19 20:27:00,Ankara,İç Anadolu,Kitap,Veri Bilimi 101,2,255.70,0.05,...,Mevcut,3.0,2.0,Hayır,0.247,545.73,2024,4,19,Friday
1390,SIP101279,MUS1149,2024-08-24 08:38:00,Ankara,İç Anadolu,Elektronik,Akıllı Saat,4,3074.80,0.15,...,Yeni,4.0,5.0,Hayır,0.414,10494.22,2024,8,24,Saturday
1393,SIP101093,MUS1187,2024-06-08 20:42:00,İstanbul,Marmara,Spor,Yoga Matı,1,772.23,0.00,...,Mevcut,3.0,2.0,Hayır,0.275,802.13,2024,6,8,Saturday


In [ ]:
# 6. aykırı değer sayılarını sütun bazında incele
for sutun in sayisal_sutunlar:

  alt_sinir = aykiri_sinirlar[sutun]["alt_sinir"]
  ust_sinir = aykiri_sinirlar[sutun]["ust_sinir"]

  aykiri_sayi = ((df[sutun] < alt_sinir) | (df[sutun] > ust_sinir)).sum()
  print(f"{sutun}: {aykiri_sayi}")

adet: 6
birim_fiyat: 92
indirim_orani: 0
kargo_ucreti: 0
teslimat_gunu: 5
musteri_puani: 151
kar_marji_orani: 0
toplam_tutar: 134
siparis_yili: 0
siparis_ayi: 0
siparis_gunu: 0


In [ ]:
# 7. aykırı değer içeren kayıtları veri setinden temizleme
df_temiz = df[~aykiri_maskesi].copy()
df_temiz.shape

(1095, 22)

In [ ]:
# 8. temizlenmiş veri setini kaydet
df_temiz.to_csv("e_ticaret_veri_seti_aykiri_degerler_duzenlendi.csv")